In [0]:
# Credentials come from the Databricks secret scope (values are redacted in output)
SECRET_SCOPE = "azure-sql-scope"
jdbc_user = dbutils.secrets.get(SECRET_SCOPE, "sql-user")
jdbc_pwd  = dbutils.secrets.get(SECRET_SCOPE, "sql-pwd")

jdbc_url = ("jdbc:sqlserver://capstone6dbserver.database.windows.net:1433;"
            "database=capstone6db;"
            "encrypt=true;"
            "trustServerCertificate=false;"
            "hostNameInCertificate=*.database.windows.net;"
            "loginTimeout=30;"
)

try:
    region = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.region_id")
        .option("user", jdbc_user)
        .option("password", jdbc_pwd)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(region)
except Exception as e:
    print(str(e)[:2000])

In [0]:
try:
    tower = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.tower_id")
        .option("user", jdbc_user)
        .option("password", jdbc_pwd)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(tower)
except Exception as e:
    print(str(e)[:2000])

In [0]:
region.write.mode("overwrite").saveAsTable("capstone6_telecom.bronze.region_id")
tower.write.mode("overwrite").saveAsTable("capstone6_telecom.bronze.tower_id")

In [0]:
BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = BRONZE   # profiling reads and writes the bronze schema
ev  = f"{cat}.bronze_network_events"

In [0]:
display(spark.sql(f"""
SELECT 'network_events' AS source, COUNT(*) AS actual, 11540 AS expected FROM {ev}
UNION ALL SELECT 'status_updates',   COUNT(*), 520 FROM {cat}.bronze_status_updates
UNION ALL SELECT 'secondary_events', COUNT(*), 310 FROM {cat}.bronze_secondary_events
UNION ALL SELECT 'region_id',        COUNT(*), 110 FROM {cat}.region_id
UNION ALL SELECT 'tower_id',         COUNT(*), 150 FROM {cat}.tower_id
"""))

In [0]:
display(spark.sql(f"SELECT source_file, COUNT(*) AS rows FROM {ev} GROUP BY source_file ORDER BY source_file"))
display(spark.sql(f"DESCRIBE TABLE {ev}"))

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS total_rows,                                   -- 11,540
       COUNT(DISTINCT measurement_id) AS unique_ids,             -- 11,500
       COUNT(*) - COUNT(DISTINCT measurement_id) AS duplicate_rows  -- 40
FROM {ev}"""))

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS exact_duplicate_groups FROM (
  SELECT measurement_id, event_timestamp, ingest_timestamp, region_id, tower_id,
         network_type, network_status, unit, latency_ms, packet_loss_pct,
         operator_id, source_system
  FROM {ev}
  GROUP BY ALL
  HAVING COUNT(*) > 1
)"""))

In [0]:
display(spark.sql(f"""
SELECT measurement_id, COUNT(*) AS copies, collect_set(source_file) AS files
FROM {ev}
GROUP BY measurement_id
HAVING COUNT(*) > 1
ORDER BY measurement_id"""))

In [0]:
cols = ["measurement_id","event_timestamp","ingest_timestamp","region_id","tower_id",
        "network_type","network_status","unit","latency_ms","packet_loss_pct",
        "operator_id","source_system"]
sel = ", ".join([f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols])
display(spark.sql(f"SELECT {sel} FROM {ev}"))

In [0]:
display(spark.sql(f"SELECT source_schema_version, COUNT(*) AS rows FROM {ev} GROUP BY 1"))

In [0]:
display(spark.sql(f"""
SELECT MIN(latency_ms) AS min_lat, PERCENTILE(latency_ms, 0.5) AS median_lat,
       PERCENTILE(latency_ms, 0.99) AS p99_lat, MAX(latency_ms) AS max_lat,
       MIN(packet_loss_pct) AS min_loss, MAX(packet_loss_pct) AS max_loss
FROM {ev}"""))

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS extreme_latency_rows           -- expect 35
FROM {ev} WHERE latency_ms > 500"""))

display(spark.sql(f"""
SELECT measurement_id, latency_ms, source_file
FROM {ev} WHERE latency_ms > 500
ORDER BY latency_ms DESC"""))

In [0]:
display(spark.sql(f"""
SELECT network_status, COUNT(*) AS rows,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM {ev} GROUP BY 1 ORDER BY rows DESC"""))
display(spark.sql(f"SELECT network_type, COUNT(*) AS rows FROM {ev} GROUP BY 1 ORDER BY 1"))
display(spark.sql(f"SELECT unit, COUNT(*) AS rows FROM {ev} GROUP BY 1"))

In [0]:
display(spark.sql(f"""
SELECT previous_status, new_status, COUNT(*) AS rows
FROM {cat}.bronze_status_updates GROUP BY 1, 2 ORDER BY rows DESC"""))

In [0]:
display(spark.sql(f"""
SELECT
 (SELECT COUNT(*) FROM {ev} e LEFT JOIN {cat}.region_id r
    ON e.region_id = r.region_id WHERE r.region_id IS NULL) AS unmatched_region_rows,
 (SELECT COUNT(*) FROM {ev} e LEFT JOIN {cat}.tower_id t
    ON e.tower_id = t.tower_id WHERE t.tower_id IS NULL) AS unmatched_tower_rows,
 (SELECT COUNT(*) FROM {cat}.bronze_status_updates s LEFT ANTI JOIN
    (SELECT DISTINCT measurement_id FROM {ev}) e
    ON s.measurement_id = e.measurement_id) AS status_rows_without_event,
 (SELECT COUNT(*) FROM {cat}.bronze_secondary_events s LEFT ANTI JOIN
    (SELECT DISTINCT measurement_id FROM {ev}) e
    ON s.measurement_id = e.measurement_id) AS secondary_rows_without_event
"""))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.profiling_summary AS
SELECT test_id, metric, actual, expected,
       CASE WHEN actual = expected THEN 'PASS' ELSE 'FAIL' END AS result,
       current_timestamp() AS checked_at
FROM (
  SELECT 'TC01' AS test_id, 'total_rows' AS metric, COUNT(*) AS actual, 11540 AS expected FROM {ev}
  UNION ALL SELECT 'TC01', 'unique_measurement_ids', COUNT(DISTINCT measurement_id), 11500 FROM {ev}
  UNION ALL SELECT 'TC01', 'duplicate_rows', COUNT(*) - COUNT(DISTINCT measurement_id), 40 FROM {ev}
  UNION ALL SELECT 'TC02', 'null_network_type', SUM(CASE WHEN network_type IS NULL THEN 1 ELSE 0 END), 12 FROM {ev}
  UNION ALL SELECT 'TC02', 'extreme_latency_rows', SUM(CASE WHEN latency_ms > 500 THEN 1 ELSE 0 END), 35 FROM {ev}
)""")
display(spark.table(f"{cat}.profiling_summary"))